# NOTEBOOK DE PRODUCCIÓN 
Predicción de demanda de fertilizantes
Carga el modelo entrenado desde MLflow, predice la demanda a partir
del pronóstico de lluvia, y escribe el resultado en gold.prediccion_demanda.
Diseñado para ejecutarse como job programado mensual.

In [0]:
import mlflow

MODEL_URI = "models:/m-acb0765edff343acbf3f900c60145ae5"
modelo_prod = mlflow.sklearn.load_model(MODEL_URI)
print("Modelo cargado desde MLflow:", MODEL_URI)

In [0]:
from pyspark.sql import functions as F
import pandas as pd

# lee el pronósticos de lluvia
clima_pd = (
    spark.table("bronze.clima_raw")
    .select("departamento", "anio", "mes", "precip_pronosticada_mm")
    .toPandas()
)

# One-hot alineado con features del modelo
features_modelo = list(modelo_prod.feature_names_in_)
clima_encoded = pd.get_dummies(clima_pd, columns=["departamento"], dtype=int)
clima_encoded = clima_encoded.rename(columns=lambda c: c.replace("departamento_", "departamento_destino_"))
for col in features_modelo:
    if col not in clima_encoded.columns:
        clima_encoded[col] = 0
X_prod = clima_encoded[features_modelo]

# se preedice y se escribe a Gold
clima_pd["demanda_predicha_tm"] = modelo_prod.predict(X_prod).round(1)
resultado_spark = spark.createDataFrame(
    clima_pd[["departamento", "anio", "mes", "precip_pronosticada_mm", "demanda_predicha_tm"]]
)
(resultado_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.prediccion_demanda"))

print(f"Predicciones escritas en gold.prediccion_demanda: {resultado_spark.count()} filas")